In [1]:
import tensorflow as tf
import os
import matplotlib.pyplot as plt

In [2]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("fahadullaha/facial-emotion-recognition-dataset")

print("Path to dataset files:", path)

/Users/saifullah/miniconda3/envs/ML/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Path to dataset files: /Users/saifullah/.cache/kagglehub/datasets/fahadullaha/facial-emotion-recognition-dataset/versions/1


In [3]:

DATASET_PATH = os.path.join(path, "processed_data")
IMG_SIZE = 48
BATCH_SIZE = 32
NUM_CLASSES = 7
path

'/Users/saifullah/.cache/kagglehub/datasets/fahadullaha/facial-emotion-recognition-dataset/versions/1'

In [4]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    color_mode="grayscale",
    label_mode='categorical'
)

# test dataset
val_ds = tf.keras.utils.image_dataset_from_directory(
    DATASET_PATH,
    validation_split=0.2,
    subset="validation",
        seed=42,                 # Same seed — same split guarantee
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    color_mode='grayscale',
    label_mode='categorical'

)

class_names = train_ds.class_names
print("Classes:", class_names)


Found 49779 files belonging to 7 classes.
Using 39824 files for training.


2026-06-26 11:04:53.295707: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M5 Pro
2026-06-26 11:04:53.295735: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2026-06-26 11:04:53.295738: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.88 GB
2026-06-26 11:04:53.295752: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-06-26 11:04:53.295764: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Found 49779 files belonging to 7 classes.
Using 9955 files for validation.
Classes: ['angry', 'disgust', 'fear', 'happy', 'neutral', 'sad', 'surprise']


In [5]:
# normalization layer, convert all train data to same scale 0 to 1
normalization_layer = tf.keras.layers.Rescaling(1./255)

# augmentation layer
augmentation_layer = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomTranslation(0.1, 0.1),
])

# Apply to datasets
train_ds = train_ds.map(lambda x, y: (augmentation_layer(x, training=True), y))
val_ds   = val_ds  # val এ কিছুই করতে হবে না!


# Performance optimization — M5 Pro তে এটা জরুরি
AUTOTUNE = tf.data.AUTOTUNE
train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
val_ds   = val_ds.cache().prefetch(buffer_size=AUTOTUNE)



In [6]:
from tensorflow.keras.metrics import Precision, Recall
from tensorflow.keras.regularizers import l2

# CNN model
model = tf.keras.Sequential([
        # Normalize first layer হিসেবে
    tf.keras.layers.Rescaling(1./255, input_shape=(IMG_SIZE, IMG_SIZE, 1)),


    # block 1
    tf.keras.layers.Conv2D(32, (3,3), padding="same", activation="relu", input_shape=(IMG_SIZE, IMG_SIZE, 1)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv2D(32, (3,3), padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(), # 48/2 = 24X24Xfilter(32)
    tf.keras.layers.Dropout(0.3),

    # block 2 -- input 24X24X32
    tf.keras.layers.Conv2D(64, (3,3), padding="same", activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv2D(64, (3,3), padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(), # 24/2 = 12
    tf.keras.layers.Dropout(0.3),
    # output = 12X12X64

    # block 3
    tf.keras.layers.Conv2D(128, (3,3), padding="same", activation="relu"),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Conv2D(128, (3,3), padding="same", activation="relu"),
    tf.keras.layers.MaxPooling2D(), # 12/2 = 6
    tf.keras.layers.Dropout(0.5),
    # output = 6X6X128

    # classifier
    tf.keras.layers.Flatten(), # 6X6X128 = 4608 values
    tf.keras.layers.Dense(256, activation='relu'), # output=inputXoutput+bias = 4608X256+256=1,179,904

    tf.keras.layers.BatchNormalization(),
    #BatchNormalization প্রতিটা feature এর জন্য 4টা value রাখে:
        # γ (gamma)   → 256  ← learnable (scale)
        # β (beta)    → 256  ← learnable (shift)
        # μ (mean)    → 256  ← non-learnable (running stats)
        # σ (variance)→ 256  ← non-learnable (running stats)
    # 256 X4 = 1024 output for this block


    tf.keras.layers.Dropout(0.6),
    tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')
    # input × output + bias
    # 256 X 7 + 7 (7 is NUM_CLASSES)
    # = 1799
])


model.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=['accuracy', Precision(name='precision'), Recall(name='recall')]
)



model.summary()


/Users/saifullah/miniconda3/envs/ML/lib/python3.12/site-packages/keras/src/layers/preprocessing/data_layer.py:95: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/Users/saifullah/miniconda3/envs/ML/lib/python3.12/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_1 (Rescaling)         │ (None, 48, 48, 1)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 48, 48, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 48, 48, 32)     │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 48, 48, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 24, 24, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 24, 24, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 24, 24, 64)     │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 24, 24, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 12, 12, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 12, 12, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 12, 12, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 12, 12, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 6, 6, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 4608)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │     1,179,904 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 256)            │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 7)              │         1,799 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,470,055 (5.61 MB)

 Trainable params: 1,469,095 (5.60 MB)

 Non-trainable params: 960 (3.75 KB)

In [ ]:
callbacks = [
    # Best model save করবে
    tf.keras.callbacks.ModelCheckpoint(
        'emotion_model_best.keras',
        save_best_only=True,
        monitor='val_accuracy',
        verbose=1
    ),
    # Overfit হলে early stop
    tf.keras.callbacks.EarlyStopping(
        patience=7,
        restore_best_weights=True,
        verbose=1
    ),
    # Val loss stuck হলে lr কমাবে
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        verbose=1
    )
]

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=50,
    callbacks=callbacks  # ← শুধু এটা add করোq
)

Epoch 1/50


/Users/saifullah/miniconda3/envs/ML/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
2026-06-26 11:04:54.376687: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


1244/1245 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.2348 - loss: 2.1551 - precision: 0.2666 - recall: 0.0529
Epoch 1: val_accuracy improved from None to 0.38122, saving model to emotion_model_best.keras

Epoch 1: finished saving model to emotion_model_best.keras
1245/1245 ━━━━━━━━━━━━━━━━━━━━ 41s 30ms/step - accuracy: 0.2349 - loss: 2.1548 - precision: 0.2668 - recall: 0.0530 - val_accuracy: 0.3812 - val_loss: 1.6386 - val_precision: 0.7073 - val_recall: 0.1008 - learning_rate: 0.0010
Epoch 2/50
1244/1245 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.3623 - loss: 1.6721 - precision: 0.6192 - recall: 0.1216
Epoch 2: val_accuracy improved from 0.38122 to 0.44571, saving model to emotion_model_best.keras

Epoch 2: finished saving model to emotion_model_best.keras
1245/1245 ━━━━━━━━━━━━━━━━━━━━ 37s 30ms/step - accuracy: 0.3623 - loss: 1.6719 - precision: 0.6196 - recall: 0.1217 - val_accuracy: 0.4457 - val_loss: 1.4833 - val_precision: 0.7044 - val_recall: 0.2104 - learning_rate: 0

In [ ]:
# ============================================
# 5. PLOT TRAINING HISTORY
# ============================================
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history.history['accuracy'],     label='Train accuracy')
ax1.plot(history.history['val_accuracy'], label='Val accuracy')
ax1.set_title('Accuracy')
ax1.legend()

ax2.plot(history.history['loss'],     label='Train loss')
ax2.plot(history.history['val_loss'], label='Val loss')
ax2.set_title('Loss')
ax2.legend()

plt.tight_layout()
plt.savefig('training_history.png')
plt.show()
print("✅ Training done! Model saved to:", DATASET_PATH)
